# Basic use of `Model4DSTEM`

Reasoning about ray paths

In [1]:
from microscope_calibration.common.model import Model4DSTEM, PixelYX
from microscope_calibration.util.sympy import lambdify

import jax
import jax.numpy as jnp
import optimistix

import sympy

We instantiate a model with values that could be found in the real world for test purposes or as a starting value in abstence of better info.

In [2]:
default_model = Model4DSTEM.default()

As an example, we define a simplified function that maps slope and overfocus to specimen pixel coordinates and detector pixel coordinates. We can define such functions as we like. This is particularly useful for working with optimizers and solvers since we can choose that way which parameters remain constant and which are variable.

We only work in y direction and don't introduce a rotation to keep it simpler in this example.

In [3]:
def specimen_detector(slope_y: float, overfocus: float):
    derived = default_model.derive(overfocus=overfocus)
    result = derived.trace(
        scan_pos=PixelYX(y=0.0, x=0.0),
        source_dy=slope_y,
        source_dx=0.0,
    )
    return (
        result["specimen"].sampling["scan_px"].y,
        result["detector"].sampling["detector_px"].y,
    )

In [4]:
specimen_detector(
    slope_y=0.01,
    overfocus=0.0001,
)

(1.00000000000000, 200.020000000000)

Now we want to find slope and overfocus that match an observed ray that passed through specific pixels of the specimen and detector.

In [5]:
specimen_target = 3
detector_target = 30

## Variant one: Symbolic, exact solution

Many thanks to [Margarita Usoltceva](https://github.com/Margus2004/) for working out how SymPy can be used with TemGym!

Define symbols for the target variables

In [6]:
slope_symbol = sympy.Symbol("slope_y")
overfocus_symbol = sympy.Symbol("overfocus")

The model is compatible with SymPy and returns an expression for the distance as a function of our symbols. Note that the remaining model parameters are simplified to constants! That means we can derive simplified expressions as we like, depending on which parameters we feed in as symbols and which as numerical constants. All SymPy symbols and expressions should work.

In [7]:
specimen_detector(slope_y=slope_symbol, overfocus=overfocus_symbol)

(1000000.0*overfocus*slope_y, 20000.0*overfocus*slope_y + 20000.0*slope_y)

We now let SymPy find a solution where both values are equal to zero

In [8]:
specimen_value, detector_value = specimen_detector(slope_y=slope_symbol, overfocus=overfocus_symbol)

In [9]:
exact_solution = sympy.solve(
    [
        sympy.Equality(specimen_value, specimen_target),
        sympy.Equality(detector_value, detector_target),
    ]
)

In [10]:
exact_solution

[{overfocus: 0.00200400801603206, slope_y: 0.00149700000000000}]

We can convert this to floats if we want to leave the world of SymPy and continue with numerical calculations

In [11]:
specimen_detector(
    slope_y=float(exact_solution[0][slope_symbol]),
    overfocus=float(exact_solution[0][overfocus_symbol]),
)

(3.00000000000000, 30.0000000000000)

As expected, the difference is in the range of numerical precision

## Variant two: Numerical solution with an optimizer

`Model4DSTEM` has to be compatible with SymPy to allow symbolic math and exact solutions, which makes it incompatible with JAX. Compatibility with CuPy etc can complicate the situation further. To resolve this, the "base" versions is always strictly written for SymPy, and `lambdify` is used to derive versions that will use the modules passed in the `modules` parameter to the decorator.

`lambdify` is an extension of SymPy's [lambdify](https://docs.sympy.org/latest/modules/utilities/lambdify.html) to work better with Jax symbol trees and complex arguments.

It can also be used as a decorator, i.e. `@lambdify`.

In [12]:
specimen_detector_jax = lambdify(modules=jnp)(specimen_detector)

Define a loss function that matches the call signature for optimistix. Note the unused second positional argument. The solver will try to minimize this function.

In [13]:
def loss(args, _):
    slope_y, overfocus = args
    res = specimen_detector_jax(slope_y=slope_y, overfocus=overfocus)
    return (res[0] - specimen_target, res[1] - detector_target)

In [14]:
loss((3, 4), None)

(11999997.0, 299970.0)

We can differentiate this automatically! That is the basis for fast optimizers.

In [15]:
jax.jacobian(loss)((0.0, 0.0), None)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


((Array(0., dtype=float64, weak_type=True),
  Array(0., dtype=float64, weak_type=True)),
 (Array(20000., dtype=float64, weak_type=True),
  Array(0., dtype=float64, weak_type=True)))

Call the optimizer

In [16]:
opt_result = optimistix.least_squares(
    fn=loss, args=None, solver=optimistix.BFGS(atol=1e-12, rtol=1e-12), y0=jnp.array((0.0, 0.0))
)

In [17]:
opt_result.value

Array([0.001497  , 0.00200401], dtype=float64)

Again, the difference is in the range of numerical precision

In [18]:
specimen_detector(slope_y=float(opt_result.value[0]), overfocus=float(opt_result.value[1]))

(3.00000000000000, 30.0000000000000)

In summary:

`Model4DSTEM` resp. TemGymCore can be seen as a tool to build up mathematical expressions or numerical functions for ray paths as a function of model parameters and arguments to `trace()`. The easiest and most understandable approach is to simply define a function that takes only the desired inputs and returns only the desired outputs, with the rest of the model coming in via a closure (like here) or an extra argument like the unused second argument to `loss2()`.

By solving for or optimizing the input parameters so that the output values match a desired value, we can reason about which ray goes where in a microscope, which settings we must apply, etc.

The key is to keep TemGymCore and `Model4DSTEM` as simple as possible in a clean, functional programming style. High performance processing routines can then be derived with e.g. `@lambdify` followed by `@jax.jit`, etc.